In [14]:
# PROGRAM 1: Automatic Headlight Control System

import cv2
import numpy as np

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions

# Load pretrained CNN
model = MobileNetV2(weights="imagenet")

# Read an image (camera sensor simulation)
image = cv2.imread("day.jpg")

# Check if image was loaded
if image is None:
    print("Image not found")
    exit()

# Resize image for CNN
img = cv2.resize(image, (224, 224))

# Prepare image for CNN
x = np.expand_dims(img, axis=0)
x = preprocess_input(x)

# CNN Prediction
predictions = model.predict(x, verbose=0)

print("CNN Prediction:")
print(decode_predictions(predictions, top=3)[0])

# Calculate ambient brightness
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
brightness = np.mean(gray)

print("Ambient Brightness:", brightness)

# Automatic Headlight Control
if brightness < 80:
    print("Headlights ON")
else:
    print("Headlights OFF")

CNN Prediction:
[('n04482393', 'tricycle', np.float32(0.12121512)), ('n03444034', 'go-kart', np.float32(0.059264287)), ('n03891251', 'park_bench', np.float32(0.042795822))]
Ambient Brightness: 143.84884640738298
Headlights OFF


In [1]:
# PROGRAM 2: Parking Assistance System

import cv2
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO("yolov8n.pt")

# Load parking image
image = cv2.imread("parking.jpg")

# Check if image was loaded
if image is None:
    print("Image not found")
    exit()

# Run object detection
results = model(image)

# Copy image
output = image.copy()

for result in results:

    boxes = result.boxes

    for box in boxes:

        # Bounding box coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Confidence
        confidence = float(box.conf[0])

        # Class ID
        class_id = int(box.cls[0])

        # Class name
        label = model.names[class_id]

        # Width and Height
        width = x2 - x1
        height = y2 - y1

        # Bounding box area
        area = width * height

        # Estimate distance using bounding box area
        if area > 120000:
            warning = "STOP!"
            color = (0, 0, 255)

        elif area > 60000:
            warning = "BRAKE NOW"
            color = (0, 140, 255)

        elif area > 25000:
            warning = "SLOW DOWN"
            color = (0, 255, 255)

        else:
            warning = "SAFE"
            color = (0, 255, 0)

        # Draw bounding box
        cv2.rectangle(
            output,
            (x1, y1),
            (x2, y2),
            color,
            2
        )

        # Object name and confidence
        cv2.putText(
            output,
            f"{label} {confidence:.2f}",
            (x1, y1 - 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2
        )

        # Warning message
        cv2.putText(
            output,
            warning,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

# Display output
cv2.imshow("Parking Assistance System", output)

cv2.waitKey(0)
cv2.destroyAllWindows()


0: 480x640 1 cell phone, 193.4ms
Speed: 6.2ms preprocess, 193.4ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)
